### Langgraph with Graph API

In [1]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

/Users/pradip/Documents/AgenticLanggraph/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/pradip/Documents/AgenticLanggraph/venv/lib/python3.9/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
class State(TypedDict):
    messages:Annotated[list, add_messages] #annoted list

graph_builder = StateGraph(State)
graph_builder

In [3]:
import os 
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.getcwd(), '.env'))

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY') or ''

In [4]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm=ChatGroq(model='llama-3.1-8b-instant')
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x341864640>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x341887d00>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
## Node functionality
def chatbot(state:State):
    return {'messages':[llm.invoke(state['messages'])]}

In [6]:
graph_builder = StateGraph(State)
graph_builder.add_node('llmchatbot', chatbot)
graph_builder.add_edge(START, 'llmchatbot')
graph_builder.add_edge('llmchatbot', END)

graph = graph_builder.compile()

In [9]:
from langchain_core.messages import HumanMessage

result = graph.invoke({'messages':[HumanMessage(content='Hi')]})
result

{'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='388db7a0-498d-4a8e-bb59-f228fe595331'),
  AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 36, 'total_tokens': 44, 'completion_time': 0.034753879, 'completion_tokens_details': None, 'prompt_time': 0.001659484, 'prompt_tokens_details': None, 'queue_time': 0.080698016, 'total_time': 0.036413363}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f0c98-769d-7671-adf1-d1509523d118-0', usage_metadata={'input_tokens': 36, 'output_tokens': 8, 'total_tokens': 44})]}